# Time Series as Dynamical Systems: Delay Embedding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/time-series/dynamical_systems_time_series.ipynb)

Takens' theorem lets you rebuild a system's attractor from a **single** measured variable
using time-delayed copies of it. This notebook builds the whole pipeline from scratch in
NumPy on the Lorenz system, then tries it on real sunspot data:

1. **Delay embedding** — reconstruct the Lorenz attractor from the x-coordinate alone.
2. **Choosing tau and m** — mutual information and false nearest neighbours.
3. **Simplex projection** — nearest-neighbour forecasting that beats a linear model.
4. **Lyapunov exponent** — the predictability horizon of a chaotic series.

Companion post: *Time Series as Dynamical Systems: Delay Embedding* on sesen.ai.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
np.random.seed(0)

## 1. The Lorenz system and delay embedding

We integrate the Lorenz system, then throw away everything except the `x` coordinate and
rebuild the attractor from delayed copies of it.

In [ ]:
def lorenz(n, dt=0.01, sigma=10.0, rho=28.0, beta=8.0 / 3.0,
           x0=(1.0, 1.0, 1.0), transient=2000):
    """Integrate the Lorenz system with RK4. Returns (n, 3) after the transient."""
    def f(s):
        x, y, z = s
        return np.array([sigma * (y - x), x * (rho - z) - y, x * y - beta * z])
    traj = np.empty((n + transient, 3))
    s = np.array(x0, float)
    for i in range(n + transient):
        traj[i] = s
        k1 = f(s); k2 = f(s + 0.5 * dt * k1)
        k3 = f(s + 0.5 * dt * k2); k4 = f(s + dt * k3)
        s = s + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
    return traj[transient:]

def time_delay_embed(x, tau, m):
    """Delay vectors [x(t), x(t-tau), ..., x(t-(m-1)tau)]. Returns (N, m)."""
    N = len(x) - (m - 1) * tau
    return np.column_stack([x[i * tau: i * tau + N] for i in range(m)])

traj = lorenz(8000)
x = traj[:, 0]                       # measure ONLY this coordinate
emb = time_delay_embed(x, tau=16, m=3)

fig = plt.figure(figsize=(11, 4.5))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot(traj[:, 0], traj[:, 1], traj[:, 2], lw=0.3, color="gray")
ax1.set_title("True attractor (x, y, z)")
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.plot(emb[:, 0], emb[:, 1], emb[:, 2], lw=0.3, color="C1")
ax2.set_title("Reconstructed from x(t) alone")
plt.tight_layout(); plt.show()

## 2. Choosing the delay tau

Take the first minimum of the average mutual information between the signal and its
delayed copy: the shortest delay that adds new information instead of a near-duplicate.

In [ ]:
def mutual_information(x, max_lag=60, bins=32):
    ami = np.empty(max_lag)
    for lag in range(1, max_lag + 1):
        c, _, _ = np.histogram2d(x[:-lag], x[lag:], bins=bins)
        p = c / c.sum()
        pa, pb = p.sum(1), p.sum(0)
        nz = p > 0
        ami[lag - 1] = np.sum(p[nz] * np.log(p[nz] / (pa[:, None] * pb[None, :])[nz]))
    return ami

def first_minimum(curve):
    for i in range(1, len(curve) - 1):
        if curve[i] < curve[i - 1] and curve[i] < curve[i + 1]:
            return i + 1
    return int(np.argmin(curve)) + 1

ami = mutual_information(x)
tau = first_minimum(ami)
print("tau =", tau, "samples")
plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(ami) + 1), ami)
plt.axvline(tau, color="C1", ls="--"); plt.xlabel("delay"); plt.ylabel("mutual info (nats)")
plt.title(f"First minimum at tau = {tau}"); plt.show()

## 3. Choosing the dimension m: false nearest neighbours

Raise `m` until neighbours stop being artefacts of a too-cramped embedding. The fraction
of false neighbours collapses to zero at the true dimension.

In [ ]:
def false_nearest_neighbours(x, tau, m_max=8, rtol=15.0, atol=2.0):
    sigma = x.std(); fractions = []
    for m in range(1, m_max + 1):
        e = time_delay_embed(x, tau, m)
        n = len(e) - tau
        e = e[:n]
        d, idx = cKDTree(e).query(e, k=2)
        nn, d_m = idx[:, 1], d[:, 1]
        d_extra = np.abs(x[np.arange(n) + m * tau] - x[nn + m * tau])
        safe = d_m > 0
        c1 = np.zeros(n, bool); c1[safe] = d_extra[safe] / d_m[safe] > rtol
        c2 = np.sqrt(d_m ** 2 + d_extra ** 2) / sigma > atol
        fractions.append((c1 | c2).mean())
    return np.array(fractions)

fnn = false_nearest_neighbours(x, tau)
for m, fr in enumerate(fnn, 1):
    print(f"m={m}: {fr*100:5.1f}% false neighbours")
plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(fnn) + 1), fnn * 100, "o-")
plt.xlabel("embedding dimension m"); plt.ylabel("false neighbours (%)")
plt.title("Collapses at the true dimension"); plt.show()

## 4. Simplex projection vs a linear model

Forecast by averaging the futures of the nearest neighbours on the reconstructed
attractor. Compare against a linear autoregression on the same delay coordinates.

In [ ]:
def simplex_forecast(x, tau, m, Tp, lib_frac=0.6):
    emb = time_delay_embed(x, tau, m); tip = (m - 1) * tau
    split = int(len(emb) * lib_frac)
    lib = np.arange(split); lib = lib[tip + lib + Tp < len(x)]
    pred = np.arange(split, len(emb)); pred = pred[tip + pred + Tp < len(x)]
    d, nbr = cKDTree(emb[lib]).query(emb[pred], k=m + 1)
    w = np.exp(-d / np.maximum(d[:, [0]], 1e-12)); w /= w.sum(1, keepdims=True)
    return np.sum(w * x[tip + lib[nbr] + Tp], axis=1), x[tip + pred + Tp]

def linear_forecast(x, tau, m, Tp, lib_frac=0.6):
    emb = time_delay_embed(x, tau, m); tip = (m - 1) * tau
    split = int(len(emb) * lib_frac)
    lib = np.arange(split); lib = lib[tip + lib + Tp < len(x)]
    pred = np.arange(split, len(emb)); pred = pred[tip + pred + Tp < len(x)]
    A = np.column_stack([emb[lib], np.ones(len(lib))])
    coef, *_ = np.linalg.lstsq(A, x[tip + lib + Tp], rcond=None)
    Ap = np.column_stack([emb[pred], np.ones(len(pred))])
    return Ap @ coef, x[tip + pred + Tp]

skill = lambda p, a: np.corrcoef(p, a)[0, 1]
horizons = [1, 5, 10, 20, 40, 80, 160]
s_simplex = [skill(*simplex_forecast(x, tau, 3, Tp)) for Tp in horizons]
s_linear = [skill(*linear_forecast(x, tau, 3, Tp)) for Tp in horizons]
plt.figure(figsize=(7.5, 4.5))
plt.plot(np.array(horizons) * 0.01, s_simplex, "o-", label="simplex (nonlinear)")
plt.plot(np.array(horizons) * 0.01, s_linear, "s-", label="linear AR")
plt.axhline(0, color="gray", lw=1); plt.xlabel("horizon (time units)")
plt.ylabel("forecast skill (rho)"); plt.legend(); plt.title("Nonlinear wins on chaos")
plt.show()

## 5. The predictability horizon (Lyapunov exponent)

Nearby trajectories separate exponentially. The slope of the log-separation is the largest
Lyapunov exponent; its reciprocal is how far ahead you can forecast.

In [ ]:
def lyapunov_rosenstein(x, tau, m, dt, max_t=150):
    emb = time_delay_embed(x, tau, m); n = len(emb)
    theiler = tau * m
    tree = cKDTree(emb); nn = np.full(n, -1)
    for i in range(n):
        _, idx = tree.query(emb[i], k=min(40, n))
        for c in idx:
            if abs(c - i) > theiler:
                nn[i] = c; break
    div = np.full(max_t, np.nan)
    for t in range(max_t):
        v = [np.log(np.linalg.norm(emb[i + t] - emb[nn[i] + t]))
             for i in range(n - t) if nn[i] >= 0 and nn[i] + t < n
             and np.linalg.norm(emb[i + t] - emb[nn[i] + t]) > 0]
        if v:
            div[t] = np.mean(v)
    return np.arange(max_t) * dt, div

times, div = lyapunov_rosenstein(x[:3000], tau, 3, dt=0.01, max_t=170)
lo, hi = 80, 150
lam = np.polyfit(times[lo:hi], div[lo:hi], 1)[0]
print(f"Lyapunov exponent ~ {lam:.2f} /time unit (Lorenz reference ~0.906)")
print(f"predictability horizon ~ {1/lam:.2f} time units")
plt.figure(figsize=(7.5, 4.5))
plt.plot(times, div); plt.axvspan(times[lo], times[hi], color="C1", alpha=0.1)
plt.xlabel("time"); plt.ylabel("mean log separation")
plt.title(f"Exponential divergence: lambda ~ {lam:.2f}"); plt.show()

## 6. Real data: sunspots

The same recipe on the yearly sunspot record. The reconstruction shows a clear cyclic
core, but real data is short and noisy, so the attractor is fuzzier than Lorenz.

In [ ]:
import statsmodels.api as sm
data = sm.datasets.sunspots.load_pandas().data
s = data["SUNACTIVITY"].values.astype(float)
tau_s = first_minimum(mutual_information(s, max_lag=20, bins=16))
print("sunspot tau =", tau_s, "years")
es = time_delay_embed(s, tau_s, 3)
fig = plt.figure(figsize=(11, 4.2))
ax1 = fig.add_subplot(1, 2, 1); ax1.plot(data["YEAR"], s)
ax1.set_title("Yearly sunspots"); ax1.set_xlabel("year")
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.plot(es[:, 0], es[:, 1], es[:, 2], lw=1, color="C1")
ax2.set_title(f"Reconstructed cycle (tau={tau_s} yr)")
plt.tight_layout(); plt.show()

## Exercises

1. **Bad delays.** Set `tau=1` and `tau=60` and re-plot the reconstruction. Watch it
   collapse onto the diagonal (too small) or fold into noise (too large).
2. **Add noise.** Add Gaussian noise to `x` before embedding. How much noise destroys the
   false-nearest-neighbours signal for the dimension `m`?
3. **Number of neighbours.** Simplex uses `m+1` neighbours. Try more (S-map style,
   distance-weighted) and see whether short-horizon skill improves.
4. **A different system.** Swap Lorenz for the Rossler system and re-run everything. What
   dimension does false nearest neighbours return?
5. **Real forecasting.** Run simplex projection on the sunspot series (leave-one-out) and
   compare its skill to a linear AR model at 1, 3, and 6 years ahead.

### References

- Takens (1981), *Detecting Strange Attractors in Turbulence*
- Sugihara & May (1990), *Nonlinear forecasting...*, Nature 344:734
- Kennel, Brown & Abarbanel (1992), *Determining embedding dimension* (false nearest neighbours)
- Rosenstein, Collins & De Luca (1993), *A practical method for calculating largest Lyapunov exponents*
- Sugihara et al. (2012), *Detecting Causality in Complex Ecosystems*, Science 338:496